In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install fastapi uvicorn nest-asyncio pyngrok transformers torch scikit-learn joblib


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import joblib

model_path = "/content/drive/MyDrive/AI/results"
label_encoder_path = "/content/drive/MyDrive/AI/label_encoder.pkl"  # rename it if needed

tokenizer = BertTokenizer.from_pretrained(model_path, local_files_only=True)
model = BertForSequenceClassification.from_pretrained(model_path, local_files_only=True)
le = joblib.load(label_encoder_path)


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import torch
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()
app = FastAPI()

class TextIn(BaseModel):
    text: str

@app.post("/predict")
def predict(text_in: TextIn):
    text = text_in.text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=1).squeeze()

    top_k_indices = torch.topk(probs, 3).indices.tolist()
    top_k_probs = probs[top_k_indices].tolist()
    top_k_labels = le.inverse_transform(top_k_indices)

    return {
        "emotions": list(zip(top_k_labels, top_k_probs))
    }


In [ ]:
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI, Request
from pydantic import BaseModel
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import joblib

# Prepare environment
nest_asyncio.apply()
app = FastAPI()

# Load model
model_path = "/content/drive/MyDrive/AI/results"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
le = joblib.load("/content/drive/MyDrive/AI/label_encoder.pkl")

# API request schema
class TextInput(BaseModel):
    text: str

# Prediction endpoint
@app.post("/predict")
def predict_emotions(item: TextInput):
    inputs = tokenizer(item.text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.nn.functional.softmax(logits, dim=1).squeeze()

    top_k_indices = torch.topk(probs, 3).indices.tolist()
    top_k_probs = probs[top_k_indices].tolist()
    top_k_labels = le.inverse_transform(top_k_indices)

    return {
        "emotions": list(zip(top_k_labels, top_k_probs))
    }


In [ ]:
# ✅ Install ngrok and uvicorn (if not already installed)
!pip install fastapi uvicorn pyngrok nest-asyncio

# ✅ Setup ngrok auth token (only once per session)
!ngrok config add-authtoken "2zrRSaFPxxdt3UUkSxlhcT4rQsT_568KqPqaNDuJMFK6v9xMX"

# ✅ Setup and run everything
from fastapi import FastAPI
from pyngrok import ngrok
from uvicorn import Config, Server
import nest_asyncio
import threading
import asyncio

# Apply asyncio patch to work inside Jupyter/Colab
nest_asyncio.apply()

# Your FastAPI app
app = FastAPI()

@app.post("/predict")
def predict(input: dict):
    text = input.get("text", "")
    return {"emotions": ["happy", "calm", "grateful"]}  # replace with real prediction logic

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"🚀 Public URL: {public_url}")

# Configure the server
config = Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = Server(config=config)

# Run the server in a background thread
def run():
    asyncio.run(server.serve())

threading.Thread(target=run).start()


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🚀 Public URL: NgrokTunnel: "https://93ffde1467c8.ngrok-free.app" -> "http://localhost:8000"


INFO:     Started server process [5348]
INFO:     Waiting for application startup.
